# Лабораторная работа № 6. PCA и кластеризация (K-means, GMM)

**Курс:** Классическое машинное обучение, 4 курс прикладной математики

## Цель работы

Применить методы снижения размерности и кластеризации, сравнить их эффективность.

**Используемые инструменты:** `numpy`, `sklearn.decomposition`, `sklearn.cluster`, `sklearn.mixture`, `matplotlib`.

### Регламент сдачи

Работа сдаётся в виде этого же ноутбука, дополненного вашим кодом. Обязательно:

1. Читаемый код с комментариями.
2. Визуализации (графики, таблицы).
3. **Текстовый вывод после каждого задания** — не только код, но и объяснение результата.
4. Финальный вывод по работе.

**Критерии оценки:** корректность реализации — 30 %, качество визуализаций и анализа — 20 %,
обоснованность выводов — 20 %, сравнение с эталонными реализациями — 15 %,
оригинальность и дополнительная работа — 15 %.

> Ячейки, помеченные `# TODO`, нужно заполнить самостоятельно.
> Ячейки с готовым кодом можно просто выполнить — они подготавливают данные и графики.

## Подготовка окружения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
sns.set_palette("viridis")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = StandardScaler().fit_transform(iris.data)
y_true = iris.target

print("Данные:", X.shape, "| классы:", np.bincount(y_true))

## Задание 1. Класс `PCA` с нуля

Алгоритм:

1. Центрирование: $\tilde{X} = X - \bar{x}$.
2. Ковариационная матрица $C = \frac{1}{n-1}\tilde{X}^T\tilde{X}$ и её собственное разложение
   (либо сразу SVD матрицы $\tilde{X}$ — численно устойчивее).
3. Главные компоненты — собственные векторы, отсортированные по убыванию собственных чисел.
4. Проекция: $Z = \tilde{X} W_k$, восстановление: $\hat{X} = Z W_k^T + \bar{x}$.

Доля объяснённой дисперсии $k$-й компоненты равна $\lambda_k / \sum_j \lambda_j$.

In [ ]:
class PCAScratch:
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.mean_ = None
        self.components_ = None            # (n_components, n_features)
        self.explained_variance_ratio_ = None

    def fit(self, X):
        # TODO:
        #   1. self.mean_ = X.mean(axis=0); Xc = X - self.mean_
        #   2. U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
        #   3. self.components_ = Vt[:self.n_components]
        #   4. доля дисперсии: s**2 / (s**2).sum()
        raise NotImplementedError

    def transform(self, X):
        # TODO: (X - self.mean_) @ self.components_.T
        raise NotImplementedError

    def inverse_transform(self, Z):
        # TODO: Z @ self.components_ + self.mean_
        raise NotImplementedError

## Задание 2. PCA на `iris`

Спроецируйте данные на две первые компоненты и раскрасьте точки по истинным классам.

In [ ]:
# TODO: постройте scatter первых двух компонент с цветом по y_true,
#       подпишите оси с долей объяснённой дисперсии, например "PC1 (72.9 %)"

In [ ]:
# TODO: постройте график накопленной объяснённой дисперсии (scree plot).
#       Сколько компонент нужно, чтобы объяснить 95 % дисперсии?

## Задание 3. Сравнение со `sklearn.decomposition.PCA`

Компоненты могут отличаться **знаком** — это нормально: собственный вектор определён
с точностью до знака. Сравнивайте по модулю или по объяснённой дисперсии.

In [ ]:
from sklearn.decomposition import PCA

# TODO: сравните explained_variance_ratio_ и абсолютные значения компонент

## Задание 4. K-means с инициализацией K-means++

Алгоритм Ллойда: чередуем присвоение объектов ближайшему центроиду и пересчёт центроидов
как средних. Минимизируется внутрикластерная сумма квадратов (inertia):

$$J = \sum_{k=1}^{K}\sum_{x \in C_k}\|x - \mu_k\|^2.$$

K-means++ выбирает начальные центры так, чтобы они были далеко друг от друга:
очередной центр берётся с вероятностью, пропорциональной $D(x)^2$ — квадрату расстояния
до ближайшего уже выбранного центра.

In [ ]:
class KMeansScratch:
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids_ = None
        self.labels_ = None
        self.inertia_ = None

    def _init_centroids(self, X):
        # TODO: реализуйте K-means++
        raise NotImplementedError

    def fit(self, X):
        # TODO: цикл «присвоить — пересчитать» до сходимости центроидов
        raise NotImplementedError

## Задание 5. Выбор числа кластеров

Посчитайте silhouette score для $K = 2, \dots, 10$. Силуэт объекта:

$$s(i) = \frac{b(i) - a(i)}{\max\bigl(a(i),\, b(i)\bigr)} \in [-1, 1],$$

где $a(i)$ — среднее расстояние до объектов своего кластера, $b(i)$ — до ближайшего чужого.

In [ ]:
from sklearn.metrics import silhouette_score

# TODO: постройте два графика рядом: inertia(K) («метод локтя») и silhouette(K).
#       Совпадают ли их рекомендации? Какое K подсказывают данные, и совпадает ли оно
#       с известным числом классов ириса (3)?

## Задание 6. GMM и сравнение с K-means

`GaussianMixture` — мягкая кластеризация: каждый объект принадлежит кластерам с вероятностями.
В отличие от K-means, кластеры могут быть вытянутыми и разного размера.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

# TODO: обучите GMM и K-means, сравните ARI с истинными метками.
#       Визуализируйте оба разбиения в пространстве первых двух главных компонент
#       (три подграфика: истинные классы | K-means | GMM)

**Вывод:** *в каких случаях GMM выигрывает у K-means? Что происходит с K-means,
если кластеры имеют разную дисперсию по разным осям?*

## Дополнительное задание. EM для GMM с диагональными матрицами

Реализуйте EM-алгоритм:

* **E-шаг:** $\gamma_{ik} = \dfrac{\pi_k\, \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_j \pi_j\, \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$
* **M-шаг:** пересчёт $\pi_k, \mu_k, \Sigma_k$ по взвешенным средним.

Ковариационные матрицы считайте диагональными. Сравните логарифм правдоподобия
со `sklearn.mixture.GaussianMixture(covariance_type="diag")`.

In [ ]:
# TODO: реализуйте EM. Подсказка: считайте всё в логарифмах и используйте
#       scipy.special.logsumexp — иначе вероятности «схлопнутся» в нуль.

## Финальный вывод

*Напишите здесь связный вывод по работе (5–10 предложений):*

- какие методы вы применили и почему;
- какие результаты получили в числах;
- где реализация «с нуля» разошлась с эталоном из `sklearn` и в чём причина;
- что бы вы улучшили, будь у вас больше времени.